# Rent by linear regression, spam by logistic regression

A friend is moving to the city and wants to know what rent to expect for an apartment of a given size. You have sixty recent listings, each with a floor area and a monthly rent. The first half of this notebook fits a line to them with gradient descent, in NumPy, and watches the loss fall. The second half aims the same machinery at a different question, whether an email is spam, by passing the weighted sum through a sigmoid and reading off a probability.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It makes both datasets from a fixed seed and prints a few rows of each.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# Sixty apartment listings: floor area in square feet, and monthly rent in dollars.
size = rng.uniform(400, 1600, 60).round()
rent = (500 + 1.5 * size + rng.normal(0, 150, 60)).round()

# Two hundred emails, the second hundred spam. Two things were counted in each one: the
# number of links, and the share of its words written in capitals, as a percentage.
links = np.concatenate([rng.poisson(1.0, 100), rng.poisson(4.0, 100)])
caps = np.concatenate([rng.normal(4, 3, 100), rng.normal(14, 6, 100)]).clip(0).round(1)
emails = np.column_stack([links, caps])   # one row per email: links, capitals
is_spam = np.repeat([0, 1], 100)           # 1 for spam

print("size and rent of the first five listings:")
print(np.column_stack([size, rent])[:5])
print("links, capitals and label of two emails of each kind:")
print(np.column_stack([emails, is_spam])[[0, 1, 100, 101]])

## Part 1 (coding task): Look at the data, then write the model and its loss

Plot before fitting anything. The cell below draws rent against size, and the points lie near a line, so the model is a line, $\hat{y} = w x + b$. The slope $w$ is the extra rent per extra square foot, and $b$ is the rent of an apartment of size zero. The lesson calls these $w_1$ and $w_0$ and folds the intercept into the weights with a column of ones. Here they stay two plain numbers.

To score a line, take every miss $\hat{y}_i - y_i$, square it so that misses above and below the line both count, and average. That is the mean squared error. The lesson sums the squares instead of averaging them. Both have their minimum at the same line, and the mean does not grow with the number of listings, so it is the one to watch.

Write `predict(x, w, b)`, which returns $w x + b$ for every entry of the array `x`, and `mse(y_hat, y)`, the mean of the squared misses.

In [ ]:
plt.scatter(size, rent, s=14)
plt.xlabel("size (square feet)")
plt.ylabel("rent (dollars per month)")
plt.show()


def predict(x, w, b):
    """The line's prediction w x + b for every entry of x."""
    # TODO
    raise NotImplementedError


def mse(y_hat, y):
    """The mean squared error of the predictions y_hat against the true values y."""
    # TODO
    raise NotImplementedError

In [ ]:
got = predict(np.array([1.0, 2.0, 3.0]), 2.0, 1.0)
assert got is not None, "predict returned None, so it is missing a return"
assert np.allclose(got, [3, 5, 7]), f"predict([1, 2, 3], w=2, b=1) should give [3, 5, 7], got {got}"
got = mse(np.array([3.0, 5.0, 7.0]), np.array([3.0, 5.0, 8.0]))
assert got is not None, "mse returned None, so it is missing a return"
assert np.isclose(got, 1 / 3), f"one miss of 1 among three points is a mean squared error of 1/3, got {got}"
assert np.isclose(mse(np.array([0.0, 0.0]), np.array([3.0, 4.0])), 12.5), "square each miss before averaging: misses of 3 and 4 give (9 + 16) / 2 = 12.5"

# A first guess, two dollars per square foot and nothing else, against the simplest model
# of all, the mean rent for everyone. The mean's error is the variance of the rents.
guess = mse(predict(size, 2.0, 0.0), rent)
flat = mse(predict(size, 0.0, rent.mean()), rent)
print(f"rent = 2 * size:    mean squared error {guess:,.0f}")
print(f"rent = mean rent:   mean squared error {flat:,.0f}")
assert np.isclose(flat, rent.var()), "predicting the mean rent for everyone should give the variance of the rents"

plt.scatter(size, rent, s=14)
plt.plot([400, 1600], predict(np.array([400.0, 1600.0]), 2.0, 0.0), color="C1", label="rent = 2 * size")
plt.axhline(rent.mean(), color="C2", label="rent = mean rent")
plt.xlabel("size (square feet)")
plt.ylabel("rent (dollars per month)")
plt.legend()
plt.show()
print("Part 1 passed")

## Part 2 (coding task): One step of gradient descent

Gradient descent lowers the loss by moving $w$ and $b$ against their partial derivatives. For the mean squared error these come out short. Write the miss of listing $i$ as $e_i = \hat{y}_i - y_i$. Then

$$
\frac{\partial L}{\partial w} = \frac{2}{n}\sum_i e_i\, x_i, \qquad \frac{\partial L}{\partial b} = \frac{2}{n}\sum_i e_i .
$$

The derivative of $e_i^2$ is $2 e_i$ times the derivative of $e_i$, which is $x_i$ for $w$ and 1 for $b$. One step with learning rate $\eta$ is $w \leftarrow w - \eta\, \partial L / \partial w$, and the same for $b$. The lesson's $\nabla L = 2X^\top(X\mathbf{w} - \mathbf{y})$ is the same gradient for the summed loss with its factor of 2, written as one matrix product.

Write `gradients(x, y, w, b)`, which returns the pair `(dw, db)`. Try it on paper first. With `x = [1, 2]`, `y = [2, 4]` and $w = b = 0$, the misses are $-2$ and $-4$, so `dw` is $2(-2 \cdot 1 - 4 \cdot 2) / 2 = -10$ and `db` is $2(-2 - 4) / 2 = -6$.

In [ ]:
def gradients(x, y, w, b):
    """The partial derivatives of mse(predict(x, w, b), y) with respect to w and b, as (dw, db)."""
    # TODO
    raise NotImplementedError

In [ ]:
got = gradients(np.array([1.0, 2.0]), np.array([2.0, 4.0]), 0.0, 0.0)
assert got is not None, "gradients returned None, so it is missing a return"
dw, db = got
assert np.isclose(dw, -10) and np.isclose(db, -6), f"on x = [1, 2], y = [2, 4] at w = b = 0 the derivatives are (-10, -6), got ({dw}, {db})"


def _rate_of_change(f, at, h):
    """How fast f changes at the point at, from two nearby values."""
    return (f(at + h) - f(at - h)) / (2 * h)


# Nudge w, then b, on the listings, and compare the change in the loss with your derivatives.
dw, db = gradients(size, rent, 1.0, 200.0)
num_dw = _rate_of_change(lambda w: mse(predict(size, w, 200.0), rent), 1.0, 1e-4)
num_db = _rate_of_change(lambda b: mse(predict(size, 1.0, b), rent), 200.0, 1e-2)
assert np.isclose(dw, num_dw, rtol=1e-4), f"dw = {dw:.3f} at w = 1, b = 200, but nudging w changes the loss at a rate of {num_dw:.3f}"
assert np.isclose(db, num_db, rtol=1e-4), f"db = {db:.3f} at w = 1, b = 200, but nudging b changes the loss at a rate of {num_db:.3f}"

# One step from w = b = 0. The learning rate has to be tiny: look at the size of dw.
dw, db = gradients(size, rent, 0.0, 0.0)
print(f"at w = b = 0:  dw = {dw:,.0f}   db = {db:,.0f}")
lr = 1e-7
w1, b1 = 0.0 - lr * dw, 0.0 - lr * db
before = mse(predict(size, 0.0, 0.0), rent)
after = mse(predict(size, w1, b1), rent)
print(f"one step at learning rate {lr}: w = {w1:.4f}, b = {b1:.6f}")
print(f"loss before the step {before:,.0f}, after it {after:,.0f}")
assert after < before, "one step against the gradient should lower the loss"
print("Part 2 passed")

## Part 3 (coding task): Feature scaling

Look at the two derivatives printed in Part 2. `dw` is about a thousand times `db`, because every size is about a thousand and the derivative for $w$ carries a factor of $x_i$. A learning rate small enough to keep $w$ stable, like the $10^{-7}$ used there, moves $b$ by less than a thousandth of a dollar per step, and $b$ has to reach about 500. Gradient descent would need millions of steps.

The fix is to change the unit of the feature. Standardizing subtracts the mean and divides by the standard deviation, so the new feature $z = (x - \bar{x}) / s$ has mean 0 and standard deviation 1. The line $w z + b$ can still be any line in the original units, and now one learning rate suits both $w$ and $b$. The check draws the loss over $(w, b)$ before and after. A long tilted valley becomes a round bowl, and gradient descent runs straight down a round bowl.

Write `standardize(x)`. Take the mean and the standard deviation with `axis=0`, so that a two-column array is standardized one column at a time, which Part 7 needs.

In [ ]:
def standardize(x):
    """x with its mean subtracted and the result divided by its standard deviation, one column at a time."""
    # TODO
    raise NotImplementedError

In [ ]:
got = standardize(np.array([1.0, 2.0, 3.0]))
assert got is not None, "standardize returned None, so it is missing a return"
assert np.allclose(got, [-1.2247, 0.0, 1.2247], atol=1e-3), f"standardize([1, 2, 3]) should give [-1.2247, 0, 1.2247], got {got}"
two = standardize(np.array([[1.0, 10.0], [2.0, 30.0], [3.0, 50.0]]))
assert two.shape == (3, 2), f"a 3 x 2 array should come back 3 x 2, got {two.shape}"
assert np.allclose(two.mean(axis=0), 0) and np.allclose(two.std(axis=0), 1), "standardize each column on its own: use axis=0"

size_z = standardize(size)
dw, db = gradients(size_z, rent, 0.0, 0.0)
print(f"at w = b = 0 on standardized sizes:  dw = {dw:,.0f}   db = {db:,.0f}")
assert abs(dw) < 10 * abs(db) and abs(db) < 10 * abs(dw), "the two derivatives should now be within a factor of ten of each other"


def _loss_grid(x, w_range, b_range):
    """The loss at every point of an 80 x 80 grid over w_range and b_range."""
    W, B = np.meshgrid(np.linspace(*w_range, 80), np.linspace(*b_range, 80))
    L = ((W[:, :, None] * x + B[:, :, None] - rent) ** 2).mean(axis=2)
    return W, B, L


fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, x, w_range, b_range, title in [
    (axes[0], size, (1.0, 2.0), (-500, 1500), "size in square feet"),
    (axes[1], size_z, (-500, 1500), (1000, 3000), "size standardized"),
]:
    W, B, L = _loss_grid(x, w_range, b_range)
    ax.contour(W, B, L, levels=np.geomspace(L.min() * 1.02, L.max(), 14))
    ax.set_title(f"loss over (w, b), {title}")
    ax.set_xlabel("w")
    ax.set_ylabel("b")
plt.tight_layout()
plt.show()
print("Part 3 passed")

## Part 4 (coding task): Train and watch the loss fall

Put the pieces together. Start at $w = b = 0$ and repeat: compute the gradient, take a step, record the loss. On the standardized sizes a learning rate of 0.1 works, and the loss should fall fast and then level off.

Write `train(x, y, lr, steps)`, which returns `(w, b, losses)`. `losses` holds the loss before the first step and after each step, so it has `steps + 1` entries. The check compares your line with the exact least-squares line. On a standardized feature the lesson's normal equations reduce to $w = \text{mean}(z\,y)$ and $b = \bar{y}$, because $\bar{z} = 0$ and $\text{std}(z) = 1$.

In [ ]:
def train(x, y, lr, steps):
    """Gradient descent from w = b = 0 at learning rate lr. Returns (w, b, losses)."""
    w, b = 0.0, 0.0
    losses = [mse(predict(x, w, b), y)]
    # TODO
    raise NotImplementedError

In [ ]:
size_z = standardize(size)
got = train(size_z, rent, 0.1, 100)
assert got is not None, "train returned None, so it is missing a return"
w_line, b_line, losses = got
losses = np.asarray(losses, dtype=float)
assert len(losses) == 101, f"100 steps give 101 losses, one before the first step and one after each step, got {len(losses)}"
assert np.isclose(losses[0], mse(predict(size_z, 0.0, 0.0), rent)), "losses[0] should be the loss at w = b = 0, before any step"
assert np.all(np.diff(losses) <= 1e-6), "at this learning rate the loss should never rise"

w_exact, b_exact = np.mean(size_z * rent), rent.mean()
print(f"gradient descent:  w = {w_line:.3f}, b = {b_line:.3f}")
print(f"least squares:     w = {w_exact:.3f}, b = {b_exact:.3f}")
assert abs(w_line - w_exact) < 0.01 and abs(b_line - b_exact) < 0.01, "after 100 steps at 0.1 the line should match least squares to two decimals"

# Back in dollars per square foot: a standardized size is (size - mean) / std.
slope = w_line / size.std()
intercept = b_line - slope * size.mean()
print(f"rent = {intercept:.0f} + {slope:.2f} * size   (the data were made from 500 + 1.5 * size, plus noise)")

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4))
left.plot(losses)
left.set_xlabel("step")
left.set_ylabel("mean squared error")
left.set_title("the loss falls")
right.scatter(size, rent, s=14)
right.plot([400, 1600], [intercept + slope * 400, intercept + slope * 1600], color="C1")
right.set_xlabel("size (square feet)")
right.set_ylabel("rent (dollars per month)")
right.set_title("the fitted line")
plt.tight_layout()
plt.show()
print("Part 4 passed")

## Part 5 (coding task): The learning rate

The learning rate sets how far each step goes. Too small, and the loss creeps down, with the line still far off when the steps run out. Too large, and each step overshoots the bottom and lands higher up the other side, so the loss grows instead of shrinking. In between, the loss falls fast and settles.

Write `sweep(x, y, rates, steps=100)`, which trains once at each rate in `rates` and returns a dictionary from each rate to its `losses`. The check runs it at 0.001, 0.1 and 1.1 for 100 steps each and draws the three curves on a log scale.

In [ ]:
def sweep(x, y, rates, steps=100):
    """The losses from train(x, y, lr, steps) for each lr in rates, as a dict keyed by the rate."""
    # TODO
    raise NotImplementedError

In [ ]:
size_z = standardize(size)
rates = (0.001, 0.1, 1.1)
curves = sweep(size_z, rent, rates)
assert curves is not None, "sweep returned None, so it is missing a return"
assert set(curves) == set(rates), f"the keys should be the rates {rates}, got {sorted(curves)}"
curves = {lr: np.asarray(losses, dtype=float) for lr, losses in curves.items()}
assert all(len(c) == 101 for c in curves.values()), "each curve should hold 101 losses, so run 100 steps"

best = mse(predict(size_z, np.mean(size_z * rent), rent.mean()), rent)
for lr in rates:
    final = curves[lr][-1]
    shown = f"{final:,.0f}" if final < 1e9 else f"{final:.1e}"
    print(f"learning rate {lr:<6} loss after 100 steps {shown}")
print(f"least squares loss {best:,.0f}")
assert curves[0.001][-1] > 10 * best, "0.001 is too small: after 100 steps the loss should still be far above the least-squares loss"
assert abs(curves[0.1][-1] - best) < 1e-3 * best, "0.1 should reach the least-squares loss"
assert curves[1.1][-1] > curves[1.1][0], "1.1 is too large: the loss should end higher than it started"

for lr in rates:
    plt.plot(curves[lr], label=f"learning rate {lr}")
plt.yscale("log")
plt.xlabel("step")
plt.ylabel("mean squared error")
plt.legend()
plt.show()
print("Part 5 passed")

## Part 6 (coding task): A sigmoid and a new loss

Now the emails. Each has two measurements, the number of links and the share of words in capitals, and a label, 1 for spam. The cell below plots them. The model is again a weighted sum, $z = w_1 x_1 + w_2 x_2 + b$, but a sum can be any number and a label is 0 or 1. The sigmoid turns the sum into a probability of spam:

$$
p = \sigma(z) = \frac{1}{1 + e^{-z}} .
$$

A large positive sum gives a probability near 1, a large negative one gives a probability near 0, and $z = 0$ gives exactly 0.5. The email is called spam when $p \ge 0.5$.

Squared error is the wrong loss for a probability, for the reasons the lesson gives. Cross-entropy charges $-\log p$ when the label is 1 and $-\log(1 - p)$ when it is 0, so a confident wrong answer costs a lot and a confident right one costs almost nothing. Averaged over the emails,

$$
J = -\frac{1}{n}\sum_i \big[\, y_i \log p_i + (1 - y_i)\log(1 - p_i)\,\big] .
$$

Write `sigmoid(z)`, which must work on arrays, and `cross_entropy(p, y)`, the mean loss of the probabilities `p` for the labels `y`.

In [ ]:
for label, name, marker in ((0, "not spam", "o"), (1, "spam", "x")):
    rows = is_spam == label
    plt.scatter(emails[rows, 0], emails[rows, 1], marker=marker, s=18, label=name)
plt.xlabel("links in the email")
plt.ylabel("words in capitals (%)")
plt.legend()
plt.show()


def sigmoid(z):
    """1 / (1 + e^(-z)), for a number or an array."""
    # TODO
    raise NotImplementedError


def cross_entropy(p, y):
    """The mean cross-entropy of the probabilities p against the labels y, which are 0 or 1."""
    # TODO
    raise NotImplementedError

In [ ]:
got = sigmoid(0.0)
assert got is not None, "sigmoid returned None, so it is missing a return"
assert np.isclose(got, 0.5), f"sigmoid(0) is 0.5, got {got}"
assert np.isclose(sigmoid(1.0), 0.7311, atol=1e-4), f"sigmoid(1) is 0.7311, got {sigmoid(1.0)}"
z = np.array([-3.0, 0.0, 3.0])
assert np.allclose(sigmoid(z), [0.0474, 0.5, 0.9526], atol=1e-4), f"sigmoid([-3, 0, 3]) should give [0.0474, 0.5, 0.9526], got {sigmoid(z)}"
assert np.allclose(sigmoid(z) + sigmoid(-z), 1), "sigmoid(-z) should equal 1 - sigmoid(z)"

got = cross_entropy(np.array([0.5, 0.5]), np.array([0, 1]))
assert got is not None, "cross_entropy returned None, so it is missing a return"
assert np.isclose(got, np.log(2)), f"a coin flip on every email costs log 2 = 0.6931 per email, got {got}"
assert np.isclose(cross_entropy(np.array([0.9, 0.1]), np.array([1, 0])), -np.log(0.9)), "0.9 on a spam and 0.1 on a real email each cost -log 0.9 = 0.1054"
assert np.isclose(cross_entropy(np.array([0.1]), np.array([1])), 2.3026, atol=1e-3), "0.1 on a spam is a confident wrong answer, and costs -log 0.1 = 2.3026"
assert np.isclose(cross_entropy(np.full(4, 0.5), np.array([0, 1, 0, 1])), np.log(2)), "average over the emails, do not sum"
print("Part 6 passed")

## Part 7 (coding task): Train the classifier

The gradient of the cross-entropy has the same shape as the one in Part 2. The lesson derives it. The error of email $i$ is $p_i - y_i$, and

$$
\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{n}\sum_i (p_i - y_i)\,\mathbf{x}_i = \frac{1}{n} X^\top(\mathbf{p} - \mathbf{y}), \qquad \frac{\partial J}{\partial b} = \frac{1}{n}\sum_i (p_i - y_i) .
$$

The factor of 2 is gone, and the sigmoid's own derivative has cancelled against the log. The lesson sums instead of averaging, so its gradient is $n$ times this one.

Both features are standardized first, with `standardize` from Part 3, so that one learning rate suits both weights. Write `logistic_gradients(X, y, w, b)`, where `X` has one row per email and `w` has one entry per column, and `train_classifier(X, y, lr, steps)`, which runs gradient descent from $\mathbf{w} = \mathbf{0}$ and $b = 0$ and returns `(w, b, losses)` as in Part 4. There is no formula for the best weights this time, so the check compares yours with Newton's method, run to convergence on the same loss.

In [ ]:
emails_z = standardize(emails)


def logistic_gradients(X, y, w, b):
    """The partial derivatives of cross_entropy(sigmoid(X @ w + b), y) with respect to w and b, as (dw, db)."""
    # TODO
    raise NotImplementedError


def train_classifier(X, y, lr, steps):
    """Gradient descent from w = 0 and b = 0 at learning rate lr. Returns (w, b, losses)."""
    w, b = np.zeros(X.shape[1]), 0.0
    losses = [cross_entropy(sigmoid(X @ w + b), y)]
    # TODO
    raise NotImplementedError

In [ ]:
got = logistic_gradients(np.array([[1.0, 0.0], [0.0, 1.0]]), np.array([1, 0]), np.zeros(2), 0.0)
assert got is not None, "logistic_gradients returned None, so it is missing a return"
dw, db = got
assert np.allclose(dw, [-0.25, 0.25]) and np.isclose(db, 0), f"at w = 0, b = 0 both probabilities are 0.5 and the errors are -0.5 and 0.5, so dw = [-0.25, 0.25] and db = 0, got ({dw}, {db})"


def _loss_at(w, b):
    return cross_entropy(sigmoid(emails_z @ w + b), is_spam)


# Nudge each weight on the emails, and compare the change in the loss with your derivatives.
w0, b0 = np.array([0.3, -0.2]), 0.1
dw, db = logistic_gradients(emails_z, is_spam, w0, b0)
for j in range(2):
    nudge = np.zeros(2)
    nudge[j] = 1e-5
    num = (_loss_at(w0 + nudge, b0) - _loss_at(w0 - nudge, b0)) / 2e-5
    assert np.isclose(dw[j], num, rtol=1e-4), f"dw[{j}] = {dw[j]:.5f}, but nudging w[{j}] changes the loss at a rate of {num:.5f}"
num = (_loss_at(w0, b0 + 1e-5) - _loss_at(w0, b0 - 1e-5)) / 2e-5
assert np.isclose(db, num, rtol=1e-4), f"db = {db:.5f}, but nudging b changes the loss at a rate of {num:.5f}"

got = train_classifier(emails_z, is_spam, 1.0, 1000)
assert got is not None, "train_classifier returned None, so it is missing a return"
w_spam, b_spam, losses = got
losses = np.asarray(losses, dtype=float)
assert len(losses) == 1001, f"1000 steps give 1001 losses, got {len(losses)}"
assert np.isclose(losses[0], np.log(2)), "before training every probability is 0.5, so the loss starts at log 2"
assert np.all(np.diff(losses) <= 1e-9), "at this learning rate the loss should never rise"

# Newton's method on the same loss, as the reference. It converges in a handful of steps.
Xa = np.column_stack([np.ones(len(is_spam)), emails_z])
theta = np.zeros(3)
for _ in range(25):
    p = 1 / (1 + np.exp(-(Xa @ theta)))
    theta = theta - np.linalg.solve(Xa.T @ (Xa * (p * (1 - p))[:, None]), Xa.T @ (p - is_spam))
print(f"gradient descent:  w = {np.round(w_spam, 3)}, b = {b_spam:.3f}")
print(f"Newton's method:   w = {np.round(theta[1:], 3)}, b = {theta[0]:.3f}")
assert np.allclose(w_spam, theta[1:], atol=0.01) and abs(b_spam - theta[0]) < 0.01, "after 1000 steps at 1.0 the weights should match Newton's method to two decimals"

p = sigmoid(emails_z @ w_spam + b_spam)
accuracy = np.mean((p >= 0.5) == is_spam)
print(f"loss {losses[-1]:.4f}, accuracy {accuracy:.3f}")
assert accuracy > 0.85, f"the classifier should get more than 85% of the emails right, got {accuracy:.3f}"

plt.plot(losses)
plt.xlabel("step")
plt.ylabel("cross-entropy")
plt.show()
print("Part 7 passed")

## Part 8 (coding task): Precision, recall and the decision boundary

Accuracy hides which mistakes a classifier makes. Call spam the positive class. A spam email flagged as spam is a true positive, a real email flagged as spam is a false positive, and a spam email let through is a false negative. Precision is the share of flagged emails that really are spam, $TP / (TP + FP)$. Recall is the share of spam that was flagged, $TP / (TP + FN)$. A filter with low precision throws away real mail, and one with low recall lets spam through.

Write `precision_recall(y, y_hat)`, which takes true labels and predicted labels, both 0 or 1, and returns the pair.

Then draw the boundary. The classifier is undecided where $z = 0$, that is where $w_1 z_1 + w_2 z_2 + b = 0$ in standardized units. That is a straight line, and solving for $z_2$ gives $z_2 = -(w_1 z_1 + b) / w_2$. Write `boundary_caps(links, w, b)`, which returns, for each number of links in the array `links`, the share of capitals at which an email sits exactly on the boundary, in the original units. Standardize `links` with the mean and standard deviation of the first column of `emails`, solve for $z_2$, then undo the standardizing of the second column.

In [ ]:
def precision_recall(y, y_hat):
    """(precision, recall) of the predicted labels y_hat against the true labels y, with 1 the positive class."""
    # TODO
    raise NotImplementedError


def boundary_caps(links, w, b):
    """The share of capitals, in percent, at which an email with each of these numbers of links has p = 0.5."""
    means, stds = emails.mean(axis=0), emails.std(axis=0)
    # TODO
    raise NotImplementedError

In [ ]:
got = precision_recall(np.array([1, 1, 0, 0, 1]), np.array([1, 0, 1, 0, 1]))
assert got is not None, "precision_recall returned None, so it is missing a return"
precision, recall = got
assert np.isclose(precision, 2 / 3), f"2 true positives among 3 flagged emails give precision 2/3, got {precision}"
assert np.isclose(recall, 2 / 3), f"2 of the 3 spam emails were flagged, so recall is 2/3, got {recall}"
precision, recall = precision_recall(np.array([1, 1, 1, 0]), np.array([1, 0, 0, 0]))
assert np.isclose(precision, 1.0) and np.isclose(recall, 1 / 3), f"one flag, correct, out of three spam: precision 1 and recall 1/3, got ({precision}, {recall})"

y_hat = (sigmoid(emails_z @ w_spam + b_spam) >= 0.5).astype(int)
precision, recall = precision_recall(is_spam, y_hat)
flagged, spam = y_hat == 1, is_spam == 1
print(f"accuracy {np.mean(y_hat == is_spam):.3f}, precision {precision:.3f}, recall {recall:.3f}")
assert np.isclose(precision, np.sum(flagged & spam) / np.sum(flagged)), "precision is the share of flagged emails that are spam"
assert np.isclose(recall, np.sum(flagged & spam) / np.sum(spam)), "recall is the share of spam that was flagged"

links_line = np.linspace(0, 10, 50)
caps_line = boundary_caps(links_line, w_spam, b_spam)
assert caps_line is not None, "boundary_caps returned None, so it is missing a return"
assert np.shape(caps_line) == (50,), f"one share of capitals for each number of links, so 50 values, got {np.shape(caps_line)}"
means, stds = emails.mean(axis=0), emails.std(axis=0)
p_line = sigmoid(((np.column_stack([links_line, caps_line]) - means) / stds) @ w_spam + b_spam)
assert np.allclose(p_line, 0.5, atol=1e-6), f"every point on the boundary should have p = 0.5, got probabilities from {p_line.min():.3f} to {p_line.max():.3f}"

for label, name, marker in ((0, "not spam", "o"), (1, "spam", "x")):
    rows = is_spam == label
    plt.scatter(emails[rows, 0], emails[rows, 1], marker=marker, s=18, label=name)
plt.plot(links_line, caps_line, color="C3", label="p = 0.5")
plt.xlabel("links in the email")
plt.ylabel("words in capitals (%)")
plt.ylim(-1, emails[:, 1].max() + 2)
plt.legend()
plt.show()
print("Part 8 passed")

## Solution

### Part 1: Look at the data, then write the model and its loss

In [ ]:
plt.scatter(size, rent, s=14)
plt.xlabel("size (square feet)")
plt.ylabel("rent (dollars per month)")
plt.show()


def predict(x, w, b):
    """The line's prediction w x + b for every entry of x."""
    return w * x + b


def mse(y_hat, y):
    """The mean squared error of the predictions y_hat against the true values y."""
    return np.mean((y_hat - y) ** 2)

### Part 2: One step of gradient descent

In [ ]:
def gradients(x, y, w, b):
    """The partial derivatives of mse(predict(x, w, b), y) with respect to w and b, as (dw, db)."""
    e = predict(x, w, b) - y
    return 2 * np.mean(e * x), 2 * np.mean(e)

### Part 3: Feature scaling

In [ ]:
def standardize(x):
    """x with its mean subtracted and the result divided by its standard deviation, one column at a time."""
    return (x - x.mean(axis=0)) / x.std(axis=0)

### Part 4: Train and watch the loss fall

In [ ]:
def train(x, y, lr, steps):
    """Gradient descent from w = b = 0 at learning rate lr. Returns (w, b, losses)."""
    w, b = 0.0, 0.0
    losses = [mse(predict(x, w, b), y)]
    for _ in range(steps):
        dw, db = gradients(x, y, w, b)
        w = w - lr * dw
        b = b - lr * db
        losses.append(mse(predict(x, w, b), y))
    return w, b, np.array(losses)

### Part 5: The learning rate

In [ ]:
def sweep(x, y, rates, steps=100):
    """The losses from train(x, y, lr, steps) for each lr in rates, as a dict keyed by the rate."""
    return {lr: train(x, y, lr, steps)[2] for lr in rates}

### Part 6: A sigmoid and a new loss

In [ ]:
for label, name, marker in ((0, "not spam", "o"), (1, "spam", "x")):
    rows = is_spam == label
    plt.scatter(emails[rows, 0], emails[rows, 1], marker=marker, s=18, label=name)
plt.xlabel("links in the email")
plt.ylabel("words in capitals (%)")
plt.legend()
plt.show()


def sigmoid(z):
    """1 / (1 + e^(-z)), for a number or an array."""
    return 1 / (1 + np.exp(-z))


def cross_entropy(p, y):
    """The mean cross-entropy of the probabilities p against the labels y, which are 0 or 1."""
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

### Part 7: Train the classifier

In [ ]:
emails_z = standardize(emails)


def logistic_gradients(X, y, w, b):
    """The partial derivatives of cross_entropy(sigmoid(X @ w + b), y) with respect to w and b, as (dw, db)."""
    e = sigmoid(X @ w + b) - y
    return X.T @ e / len(y), np.mean(e)


def train_classifier(X, y, lr, steps):
    """Gradient descent from w = 0 and b = 0 at learning rate lr. Returns (w, b, losses)."""
    w, b = np.zeros(X.shape[1]), 0.0
    losses = [cross_entropy(sigmoid(X @ w + b), y)]
    for _ in range(steps):
        dw, db = logistic_gradients(X, y, w, b)
        w = w - lr * dw
        b = b - lr * db
        losses.append(cross_entropy(sigmoid(X @ w + b), y))
    return w, b, np.array(losses)

### Part 8: Precision, recall and the decision boundary

In [ ]:
def precision_recall(y, y_hat):
    """(precision, recall) of the predicted labels y_hat against the true labels y, with 1 the positive class."""
    tp = np.sum((y_hat == 1) & (y == 1))
    fp = np.sum((y_hat == 1) & (y == 0))
    fn = np.sum((y_hat == 0) & (y == 1))
    return tp / (tp + fp), tp / (tp + fn)


def boundary_caps(links, w, b):
    """The share of capitals, in percent, at which an email with each of these numbers of links has p = 0.5."""
    means, stds = emails.mean(axis=0), emails.std(axis=0)
    z1 = (links - means[0]) / stds[0]
    z2 = -(w[0] * z1 + b) / w[1]
    return z2 * stds[1] + means[1]